# 16. RNN/LSTM 실습 — 순서가 있는 데이터

> **제16장** · **이론편 대응: 13장 (RNN/LSTM)**
> **예상 소요**: 60분
> **필요 사양**: **[CPU]** 로 실행 가능
> **다운로드**: 없음 (데이터를 코드로 생성)

---

## 이 장에서 하는 일

15장의 이미지와 달리, 이번에는 **순서가 의미를 갖는 데이터**를 다룬다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 시퀀스 데이터 만들기 | 13.1절 |
| 2 | **RNN 직접 구현** | 13.2절 |
| 3 | 그래디언트 감쇠 직접 측정 | 13.3절 |
| 4 | **LSTM 게이트 손계산 검증 (c=0.924)** ★ | 13.4절 |
| 5 | PyTorch RNN/LSTM/GRU | 13.4절 |
| 6 | 시계열 예측 실습 | 13.5절 |
| 7 | RNN의 한계 → Attention 필요성 | 13.6절 |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

---

## 1. 시퀀스 데이터 — 이론편 13.1절

이론편 13.1절에서 "문장마다 길이가 다르다"는 문제를 다뤘다. 이미지와 무엇이 다른지 정리하면 이렇다.

| 구분 | 이미지 (15장) | 시퀀스 (이 장) |
|---|---|---|
| 크기 | 모두 28×28로 고정 | 길이가 제각각 |
| 순서 | 위치가 고정 | **순서가 의미를 바꿈** |
| 입력 모양 | `(배치, 채널, H, W)` | `(배치, 시점, 특성)` |

"나는 밥을 먹었다"와 "밥을 나는 먹었다"는 같은 단어인데 뜻이 다르다.
이미지의 픽셀을 섞으면 알아볼 수 없게 되는 것과는 다른 종류의 순서 의존이다.

### 이 장에서 쓸 데이터

외부 다운로드 없이 **사인파에 잡음을 섞어** 만든다. 참값을 알고 있어야 결과를 판단할 수 있기 때문이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 사인파 + 잡음 (참값을 알고 있는 시계열)
rng = np.random.RandomState(0)
T = 1000
t = np.linspace(0, 100, T)
series = np.sin(0.2 * t) + 0.1 * rng.randn(T)

print("=" * 55)
print("시계열 데이터")
print("=" * 55)
print(f"길이     : {T}")
print(f"값 범위  : {series.min():.3f} ~ {series.max():.3f}")
print(f"주기     : 약 {2*np.pi/0.2:.1f} (시간 단위)")
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].plot(t, series, linewidth=1, color="#1E40AF")
axes[0].set_title("전체 (1000 시점)")
axes[0].set_xlabel("시간")
axes[0].grid(alpha=0.3)

axes[1].plot(t[:150], series[:150], marker="o", markersize=3,
             linewidth=1.5, color="#EA580C")
axes[1].set_title("앞부분 확대 (150 시점)")
axes[1].set_xlabel("시간")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("목표: 앞의 20개 값을 보고 다음 1개 값을 예측한다.")

In [ ]:
import numpy as np
import torch


def make_sequences(data, seq_len):
    # 시계열을 (입력 시퀀스, 다음 값) 쌍으로 자른다.
    #
    # 예: data=[1,2,3,4,5], seq_len=3 이면
    # X=[[1,2,3],[2,3,4]]  y=[4,5]
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)


SEQ_LEN = 20
X_np, y_np = make_sequences(series, SEQ_LEN)

print("=" * 55)
print("시퀀스로 자르기")
print("=" * 55)
print(f"원본 길이   : {len(series)}")
print(f"시퀀스 길이 : {SEQ_LEN}")
print(f"만들어진 쌍 : {len(X_np)}개   ({len(series)} - {SEQ_LEN})")
print()
print(f"X 모양 : {X_np.shape}")
print(f"y 모양 : {y_np.shape}")
print()
print("첫 번째 쌍")
print(f"  입력 (앞 5개만): {X_np[0][:5].round(3)} ...")
print(f"  정답           : {y_np[0]:.3f}")
print()

# PyTorch용으로 변환 — (배치, 시점, 특성) 3차원이 필요하다
X = torch.tensor(X_np, dtype=torch.float32).unsqueeze(-1)
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(-1)
print(f"텐서 모양: X {tuple(X.shape)}  ← (배치, 시점, 특성)")
print(f"          y {tuple(y.shape)}")
print()

# 시간 순서를 지켜 분할 — 섞으면 안 된다
n_train = 800
X_train, y_train = X[:n_train], y[:n_train]
X_test, y_test = X[n_train:], y[n_train:]
print(f"훈련 {len(X_train)}개 / 시험 {len(X_test)}개")
print()
print("주의: 시계열은 무작위로 섞어 나누면 안 된다.")
print("  미래 데이터로 과거를 예측하는 셈이 되어 성능이 부풀려진다.")
print("  07장에서 다룬 데이터 누수의 한 형태다.")

---

## 2. RNN 직접 구현 — 이론편 13.2절

RNN의 핵심은 **같은 계산을 시점마다 반복하며 상태를 물려준다**는 것이다.

$$h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b)$$

$h_{t-1}$이 다시 들어간다는 점이 전부다. 이 하나가 "이전까지의 정보"를 나르는 역할을 한다.

In [ ]:
import numpy as np


def rnn_forward_manual(inputs, W_xh, W_hh, b, h0=None):
    # RNN 순전파를 직접 구현 (이론편 13.2절)
    #
    # inputs: (시점, 입력차원)
    # 반환  : 모든 시점의 은닉 상태
    T, _ = inputs.shape
    hidden_size = W_hh.shape[0]
    h = np.zeros(hidden_size) if h0 is None else h0.copy()

    states = []
    for t in range(T):
        # 같은 가중치를 매 시점 반복 사용한다
        h = np.tanh(inputs[t] @ W_xh + h @ W_hh + b)
        states.append(h.copy())
    return np.array(states)


# 작은 예제로 동작 확인
rng = np.random.RandomState(0)
T, D, H = 5, 1, 3
inputs = rng.randn(T, D)
W_xh = rng.randn(D, H) * 0.5
W_hh = rng.randn(H, H) * 0.5
b = np.zeros(H)

states = rnn_forward_manual(inputs, W_xh, W_hh, b)

print("=" * 60)
print("RNN 순전파 직접 구현")
print("=" * 60)
print(f"입력: {T}시점, 은닉 크기 {H}")
print()
print(f"{'시점':<8}{'입력':<12}{'은닉 상태'}")
print("-" * 60)
for t in range(T):
    print(f"{t:<8}{inputs[t][0]:<12.4f}{states[t].round(4)}")
print("-" * 60)
print()
print("각 시점의 은닉 상태가 이전 상태의 영향을 받는다.")
print("가중치는 시점마다 같은 것을 재사용한다 — 이것이 순환의 핵심이다.")

### PyTorch RNN과 대조

직접 만든 것과 PyTorch가 같은 계산을 하는지 확인한다.
PyTorch는 편향이 두 개(`bias_ih`, `bias_hh`)라는 점만 주의하면 된다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
rnn = nn.RNN(input_size=1, hidden_size=3, batch_first=True)

# PyTorch의 가중치를 꺼내 직접 구현에 넣는다
W_ih = rnn.weight_ih_l0.detach().numpy()      # (3, 1)
W_hh_t = rnn.weight_hh_l0.detach().numpy()    # (3, 3)
b_ih = rnn.bias_ih_l0.detach().numpy()
b_hh = rnn.bias_hh_l0.detach().numpy()

# 직접 구현 (전치에 주의)
states_manual = rnn_forward_manual(
    inputs, W_ih.T, W_hh_t.T, b_ih + b_hh)

# PyTorch
x_t = torch.tensor(inputs, dtype=torch.float32).unsqueeze(0)
with torch.no_grad():
    out, _ = rnn(x_t)
states_torch = out.squeeze(0).numpy()

print("=" * 60)
print("직접 구현 vs PyTorch RNN")
print("=" * 60)
print("직접 구현 (마지막 시점)")
print(f"  {states_manual[-1].round(6)}")
print("PyTorch (마지막 시점)")
print(f"  {states_torch[-1].round(6)}")
print()
print(f"최대 차이: {np.abs(states_manual - states_torch).max():.2e}")
assert np.allclose(states_manual, states_torch, atol=1e-5)
print("[OK] 두 구현이 일치")
print()
print("PyTorch는 편향이 두 개(bias_ih, bias_hh)인데,")
print("실제로는 더해지므로 하나로 합쳐도 결과가 같다.")

---

## 3. 그래디언트 감쇠 직접 측정 — 이론편 13.3절

이론편 13.3절에서 "전언 게임처럼 신호가 감쇠한다"고 했다. **실제로 측정해 보자.**

시퀀스 마지막의 출력이 **각 시점 입력**에 대해 얼마나 민감한지 재면 된다.
PyTorch의 autograd로 구할 수 있다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt


def measure_gradient_flow(cell_type, seq_len, hidden=16, seed=0):
    """각 시점 입력에 대한 최종 출력의 그래디언트 크기를 잰다"""
    torch.manual_seed(seed)
    if cell_type == "RNN":
        rnn = nn.RNN(1, hidden, batch_first=True)
    else:
        # ── nn.LSTM 파라미터 ─────────────────────────────────────────
        #   input_size    입력 특성 차원.  **필수**
        #   hidden_size   은닉 상태 차원.  **필수**.  예: 64, 128, 256
        #   num_layers    층 수.  기본값 1.  예: 2~3 (깊게 쌓을 때)
        #   bias          기본값 True
        #   batch_first   입력 모양.  기본값 False
        #                 False: (시퀀스, 배치, 특성)
        #                 True : (배치, 시퀀스, 특성)  ← 직관적이라 권장
        #   dropout       층 사이 드롭아웃.  기본값 0
        #                 num_layers > 1 일 때만 적용된다
        #   bidirectional 양방향.  기본값 False
        #                 True 면 출력 차원이 2배가 된다
        #
        #   반환: output, (h_n, c_n)
        #     output = 모든 시점의 은닉 상태
        #     h_n    = 마지막 은닉 상태,  c_n = 마지막 셀 상태
        # ──────────────────────────────────────────────────────────────
        rnn = nn.LSTM(1, hidden, batch_first=True)
    fc = nn.Linear(hidden, 1)

    x = torch.randn(1, seq_len, 1, requires_grad=True)
    out, _ = rnn(x)
    y = fc(out[:, -1])       # 마지막 시점만 사용
    y.backward()

    return x.grad.abs().squeeze().numpy()


SEQ = 60
grad_rnn = measure_gradient_flow("RNN", SEQ)
grad_lstm = measure_gradient_flow("LSTM", SEQ)

print("=" * 60)
print(f"시점별 그래디언트 크기 (길이 {SEQ})")
print("=" * 60)
print(f"{'위치':<20}{'RNN':<18}{'LSTM'}")
print("-" * 60)
for name, idx in [("마지막 시점", -1), ("중간 시점", SEQ//2), ("첫 시점", 0)]:
    print(f"{name:<20}{grad_rnn[idx]:<18.3e}{grad_lstm[idx]:.3e}")
print("-" * 60)
print(f"{'첫/마지막 비율':<20}{grad_rnn[0]/grad_rnn[-1]:<18.1e}{grad_lstm[0]/grad_lstm[-1]:.1e}")
print()
print("이론편 13.3절에서 말한 감쇠가 실제로 일어난다.")
print("마지막 시점은 그래디언트가 크지만, 앞쪽으로 갈수록 급격히 작아진다.")

fig, ax = plt.subplots(figsize=(9, 4.5))
positions = np.arange(SEQ)
ax.semilogy(positions, grad_rnn + 1e-30, marker="o", markersize=3,
            label="RNN", color="#EA580C", linewidth=1.5)
ax.semilogy(positions, grad_lstm + 1e-30, marker="s", markersize=3,
            label="LSTM", color="#0D9488", linewidth=1.5)
ax.set_xlabel("시점 (0 = 가장 오래된 입력)")
ax.set_ylabel("그래디언트 크기 (로그 눈금)")
ax.set_title("시점별 그래디언트 — 앞쪽일수록 작아진다")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

### 이 실험을 정확히 읽는 법

그래프를 보면 **RNN과 LSTM 모두** 앞쪽으로 갈수록 그래디언트가 작아진다.
"LSTM은 감쇠가 없다"고 오해하기 쉬운데, 그렇지 않다.

**LSTM의 장점은 감쇠를 없애는 것이 아니라, 필요할 때 줄이지 않을 수 있는 통로를 갖는 것**이다.

- RNN: 가중치가 고정되어 있어 감쇠 정도를 조절할 수 없다
- LSTM: 망각 게이트를 1에 가깝게 **학습하면** 신호를 유지할 수 있다

위 실험은 **초기화 직후**라 게이트가 아직 아무것도 학습하지 않은 상태다.
그래서 차이가 크지 않게 나온다. 다음 절에서 게이트가 무엇을 하는지 직접 계산해 본다.

---

## 4. LSTM 게이트 — 이론편 13.4절 값 검증 ★

이론편 13.4절에서 LSTM의 한 시점을 손으로 계산했다. 그 값을 확인한다.

**이론편의 설정**

$$c_{t-1} = 0.8, \quad h_{t-1} = 0.3, \quad x_t = 1.0$$

**이론편에서 구한 값**

| 게이트 | 계산 | 값 |
|---|---|---|
| 망각 $f_t$ | $\sigma(0.5 \times 1.0 + 0.2 \times 0.3)$ | 0.637 |
| 입력 $i_t$ | $\sigma(0.6 \times 1.0 + 0.3 \times 0.3)$ | 0.666 |
| 후보 $\tilde{c}_t$ | $\tanh(0.7 \times 1.0 + 0.1 \times 0.3)$ | 0.623 |
| 출력 $o_t$ | $\sigma(0.4 \times 1.0 + 0.5 \times 0.3)$ | 0.634 |
| **셀 상태 $c_t$** | $0.637 \times 0.8 + 0.666 \times 0.623$ | **0.924** |
| 은닉 $h_t$ | $0.634 \times \tanh(0.924)$ | 0.462 |

In [ ]:
import numpy as np


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def lstm_step(x, h_prev, c_prev, weights):
    # LSTM 한 시점 계산 (이론편 13.4절)
    Wf, Uf, Wi, Ui, Wc, Uc, Wo, Uo = weights

    f = sigmoid(Wf * x + Uf * h_prev)      # 망각 게이트
    i = sigmoid(Wi * x + Ui * h_prev)      # 입력 게이트
    c_tilde = np.tanh(Wc * x + Uc * h_prev)  # 후보 값
    o = sigmoid(Wo * x + Uo * h_prev)      # 출력 게이트

    c = f * c_prev + i * c_tilde           # 셀 상태 갱신
    h = o * np.tanh(c)                     # 은닉 상태

    return h, c, {"f": f, "i": i, "c_tilde": c_tilde, "o": o}


# 이론편 13.4절과 완전히 같은 값
c_prev, h_prev, x = 0.8, 0.3, 1.0
weights = (0.5, 0.2,   # Wf, Uf
           0.6, 0.3,   # Wi, Ui
           0.7, 0.1,   # Wc, Uc
           0.4, 0.5)   # Wo, Uo

h, c, gates = lstm_step(x, h_prev, c_prev, weights)

print("=" * 65)
print("이론편 13.4절 값 검증")
print("=" * 65)
print(f"입력: c_prev={c_prev}, h_prev={h_prev}, x={x}")
print()
print(f"{'항목':<16}{'계산 과정':<30}{'값':<12}{'이론편'}")
print("-" * 65)
print(f"{'망각 f':<16}{'sigmoid(0.5*1.0 + 0.2*0.3)':<30}{gates['f']:<12.4f}0.637")
print(f"{'입력 i':<16}{'sigmoid(0.6*1.0 + 0.3*0.3)':<30}{gates['i']:<12.4f}0.666")
print(f"{'후보 c~':<16}{'tanh(0.7*1.0 + 0.1*0.3)':<30}{gates['c_tilde']:<12.4f}0.623")
print(f"{'출력 o':<16}{'sigmoid(0.4*1.0 + 0.5*0.3)':<30}{gates['o']:<12.4f}0.634")
print("-" * 65)
print(f"{'셀 상태 c':<16}{'f*c_prev + i*c~':<30}{c:<12.4f}0.924")
print(f"{'은닉 h':<16}{'o * tanh(c)':<30}{h:<12.4f}0.462")
print("-" * 65)

assert abs(gates["f"] - 0.637) < 0.001
assert abs(gates["i"] - 0.666) < 0.001
assert abs(c - 0.924) < 0.001
assert abs(h - 0.462) < 0.001
print("[OK] 이론편 13.4절 손계산과 일치")

In [ ]:
import numpy as np

print("=" * 60)
print("셀 상태 갱신을 나눠 보기")
print("=" * 60)

keep = gates["f"] * c_prev
add = gates["i"] * gates["c_tilde"]

print(f"c_t = {gates['f']:.4f} x {c_prev}  +  {gates['i']:.4f} x {gates['c_tilde']:.4f}")
print(f"    = {keep:.4f}          +  {add:.4f}")
print(f"    = {c:.4f}")
print()
print("각 항이 뜻하는 것")
print(f"  앞 항: 이전 기억을 {gates['f']*100:.0f}% 유지  (나머지 {(1-gates['f'])*100:.0f}%는 잊음)")
print(f"  뒤 항: 새 정보를 {gates['i']*100:.0f}%만큼 반영")
print()
print("게이트 값이 0~1 사이라는 점이 결정적이다.")
print("  0에 가까우면 완전 차단, 1에 가까우면 완전 통과")
print()
print("=" * 60)
print("망각 게이트 값에 따른 기억 유지 (이론편 13.3절)")
print("=" * 60)
print(f"{'망각 게이트':<16}{'10시점 후':<20}{'50시점 후'}")
print("-" * 60)
for f_val in [0.5, 0.9, 0.99, 1.0]:
    print(f"{f_val:<16}{f_val**10:<20.6f}{f_val**50:.6f}")
print("-" * 60)
print()
print("RNN은 가중치가 고정이라 감쇠를 조절할 수 없지만,")
print("LSTM은 이 값을 학습으로 정한다. 중요한 정보라면 1에 가깝게 만들 수 있다.")
print("곱해지는 값이 1이면 아무리 곱해도 줄지 않는다 — 이것이 장기 기억의 열쇠다.")

---

## 5. PyTorch의 RNN / LSTM / GRU

세 가지를 비교한다. 사용법은 거의 같고 **파라미터 수만 다르다.**

| 구조 | 게이트 | 파라미터 배수 |
|---|---|---|
| RNN | 없음 | 1배 |
| GRU | 2개 (업데이트·리셋) | 3배 |
| LSTM | 3개 (망각·입력·출력) | 4배 |

LSTM이 4배인 이유는 게이트 3개 + 후보값 1개에 대해 각각 가중치가 필요하기 때문이다.

In [ ]:
import torch
import torch.nn as nn

INPUT, HIDDEN = 3, 4

print("=" * 60)
print("파라미터 수 비교")
print("=" * 60)
print(f"입력 {INPUT}, 은닉 {HIDDEN} 기준")
print()
print(f"{'구조':<10}{'파라미터':<14}{'계산식':<30}{'배수'}")
print("-" * 60)

base = INPUT*HIDDEN + HIDDEN*HIDDEN + 2*HIDDEN

for name, cls, mult in [("RNN", nn.RNN, 1), ("GRU", nn.GRU, 3), ("LSTM", nn.LSTM, 4)]:
    layer = cls(INPUT, HIDDEN, batch_first=True)
    n = sum(p.numel() for p in layer.parameters())
    formula = f"{mult} x ({INPUT}x{HIDDEN} + {HIDDEN}x{HIDDEN} + {2*HIDDEN})"
    print(f"{name:<10}{n:<14}{formula:<30}{mult}배")
    assert n == mult * base

print("-" * 60)
print()
print("LSTM 내부 가중치 구조")
lstm = nn.LSTM(INPUT, HIDDEN, batch_first=True)
for name, p in lstm.named_parameters():
    print(f"  {name:<18}{str(tuple(p.shape)):<12}")
print()
print("weight_ih_l0 의 첫 차원이 16인 이유: 4개(망각·입력·후보·출력) x 은닉 4")
print("PyTorch는 네 게이트의 가중치를 하나로 합쳐서 관리한다 (계산 효율).")

In [ ]:
import torch
import torch.nn as nn

print("=" * 60)
print("출력 형태 이해하기")
print("=" * 60)

x = torch.randn(2, 7, 3)      # (배치 2, 시점 7, 특성 3)
print(f"입력: {tuple(x.shape)}   ← (배치, 시점, 특성)")
print()

rnn = nn.RNN(3, 4, batch_first=True)
out, h_n = rnn(x)
print("RNN")
print(f"  out : {tuple(out.shape)}   ← 모든 시점의 은닉 상태")
print(f"  h_n : {tuple(h_n.shape)}   ← 마지막 시점만 (층, 배치, 은닉)")
print()

lstm = nn.LSTM(3, 4, batch_first=True)
out, (h_n, c_n) = lstm(x)
print("LSTM — 상태가 두 개다")
print(f"  out : {tuple(out.shape)}")
print(f"  h_n : {tuple(h_n.shape)}   ← 은닉 상태")
print(f"  c_n : {tuple(c_n.shape)}   ← 셀 상태 (이론편 13.4절의 c)")
print()

# out의 마지막과 h_n이 같은지 확인
print(f"out[:, -1] 과 h_n[0] 이 같은가: {torch.allclose(out[:, -1], h_n[0])}")
print()
print("분류·예측에는 보통 out[:, -1] (마지막 시점)을 쓴다.")
print("모든 시점의 출력이 필요한 경우(예: 품사 태깅)는 out 전체를 쓴다.")

---

## 6. 시계열 예측 실습 — 이론편 13.5절

1절에서 만든 사인파로 실제 예측을 해 본다. 세 구조를 같은 조건에서 비교한다.

In [ ]:
import torch
import torch.nn as nn
import time


class SeqPredictor(nn.Module):
    # 시퀀스를 받아 다음 값 하나를 예측하는 모델

    def __init__(self, cell="LSTM", hidden=32):
        super().__init__()
        cells = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}
        self.rnn = cells[cell](input_size=1, hidden_size=hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1])       # 마지막 시점만 사용


def train_model(cell, epochs=200, lr=0.01, seed=42):
    torch.manual_seed(seed)
    model = SeqPredictor(cell).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()

    Xtr, ytr = X_train.to(device), y_train.to(device)
    Xte, yte = X_test.to(device), y_test.to(device)

    history = []
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        loss = crit(model(Xtr), ytr)
        loss.backward()
        opt.step()

        if ep % 20 == 0:
            model.eval()
            with torch.no_grad():
                val = crit(model(Xte), yte).item()
            history.append((ep, loss.item(), val))

    model.eval()
    with torch.no_grad():
        final_val = crit(model(Xte), yte).item()
    return model, history, final_val, time.time() - t0


print("=" * 65)
print("시계열 예측 — 세 구조 비교 (200 에폭)")
print("=" * 65)
print(f"{'구조':<10}{'파라미터':<14}{'검증 손실':<16}{'시간'}")
print("-" * 65)

models = {}
for cell in ["RNN", "GRU", "LSTM"]:
    m, hist, val, sec = train_model(cell)
    models[cell] = (m, hist, val)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{cell:<10}{n_params:<14,}{val:<16.6f}{sec:.1f}초")

print("-" * 65)
print()
print("이 문제는 주기가 짧고 규칙적이라 셋 다 잘 맞힌다.")
print("차이가 드러나려면 훨씬 긴 의존 관계가 필요하다 (7절 참조).")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- 왼쪽: 학습 곡선 ---
ax = axes[0]
colors = {"RNN": "#EA580C", "GRU": "#7C3AED", "LSTM": "#0D9488"}
for cell, (m, hist, val) in models.items():
    eps = [h[0] for h in hist]
    vals = [h[2] for h in hist]
    ax.plot(eps, vals, label=cell, linewidth=2, color=colors[cell])
ax.set_xlabel("에폭")
ax.set_ylabel("검증 손실")
ax.set_yscale("log")
ax.set_title("학습 곡선")
ax.legend()
ax.grid(alpha=0.3, which="both")

# --- 오른쪽: 예측 결과 ---
ax = axes[1]
model_lstm = models["LSTM"][0]
model_lstm.eval()
with torch.no_grad():
    pred = model_lstm(X_test.to(device)).cpu().numpy().ravel()

true = y_test.numpy().ravel()
show = 100
ax.plot(true[:show], label="실제", linewidth=2, color="#1E40AF")
ax.plot(pred[:show], label="LSTM 예측", linewidth=2,
        linestyle="--", color="#EA580C")
ax.set_xlabel("시점 (시험 데이터)")
ax.set_ylabel("값")
ax.set_title("예측 결과")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

mae = np.abs(pred - true).mean()
print(f"평균 절대 오차: {mae:.4f}")
print(f"데이터에 섞은 잡음 크기: 0.1")
print()
print("오차가 잡음 수준에 가깝다면 잘 학습된 것이다.")
print("잡음까지 맞히려는 것은 오히려 과대적합이다 (이론편 8.5절).")

---

## 7. RNN의 한계 — 이론편 13.6절

이론편 13.6절에서 RNN 계열의 두 가지 한계를 다뤘다.

### 한계 1: 병렬화가 안 된다

RNN은 $h_t$를 구하려면 $h_{t-1}$이 있어야 한다. **순서대로 계산할 수밖에 없다.**
CNN은 모든 위치를 동시에 계산할 수 있었던 것과 대조된다.

시퀀스 길이에 따라 계산 시간이 어떻게 변하는지 재 보자.

In [ ]:
import torch
import torch.nn as nn
import time

print("=" * 60)
print("시퀀스 길이에 따른 계산 시간 (이론편 13.6절)")
print("=" * 60)

lstm = nn.LSTM(16, 64, batch_first=True)
conv = nn.Conv1d(16, 64, kernel_size=3, padding=1)

print(f"{'길이':<10}{'LSTM':<18}{'CNN(1D)':<18}{'배수'}")
print("-" * 60)

for L in [50, 100, 200, 400]:
    x = torch.randn(32, L, 16)

    t0 = time.time()
    with torch.no_grad():
        _ = lstm(x)
    t_lstm = time.time() - t0

    x_conv = x.transpose(1, 2)     # Conv1d는 (배치, 채널, 길이)
    t0 = time.time()
    with torch.no_grad():
        _ = conv(x_conv)
    t_conv = time.time() - t0

    print(f"{L:<10}{t_lstm*1000:<18.2f}{t_conv*1000:<18.2f}{t_lstm/t_conv:.1f}배")

print("-" * 60)
print()
print("LSTM은 길이에 거의 비례해 시간이 늘어난다 — 순차 처리 때문이다.")
print("CNN은 병렬 계산이 가능해 길이가 늘어도 시간이 덜 는다.")
print()
print("이것이 이론편 21~22장에서 Transformer가 등장한 이유 중 하나다.")
print("Self-Attention은 모든 시점을 동시에 계산할 수 있다.")

### 한계 2: 먼 과거의 정보가 흐려진다

3절에서 그래디언트가 앞쪽으로 갈수록 작아지는 것을 봤다.
LSTM이 이를 완화하지만 **완전히 없애지는 못한다.**

이론편 13.6절에서 다룬 정보 병목도 같은 문제다. Seq2Seq에서 인코더가 문장 전체를
**고정 크기 벡터 하나**로 압축하는데, 문장이 길수록 정보 손실이 커진다.

$$\text{"나는 어제 친구와 함께 도서관에서 책을 읽었다"} \;\to\; \underbrace{[0.3, -0.7, \ldots]}_{\text{고정 크기}}$$

**Attention(이론편 21장)은 이 압축을 하지 않는다.** 모든 시점의 정보를 남겨 두고,
필요할 때 골라 본다. 다음에 이어질 실습들에서 이를 직접 구현한다.

In [ ]:
import torch
import torch.nn as nn

print("=" * 60)
print("정보 병목 — 길이가 달라도 같은 크기로 압축된다")
print("=" * 60)

encoder = nn.LSTM(16, 64, batch_first=True)

print(f"{'입력 시퀀스 길이':<20}{'인코더 출력(h_n) 크기'}")
print("-" * 60)
for L in [5, 20, 100, 500]:
    x = torch.randn(1, L, 16)
    with torch.no_grad():
        out, (h_n, c_n) = encoder(x)
    print(f"{L:<20}{tuple(h_n.shape)}")
print("-" * 60)
print()
print("입력이 5개든 500개든 최종 상태는 항상 (1, 1, 64)다.")
print("정보량이 100배 차이 나는데 담는 그릇은 같다.")
print()
print("→ 이론편 17.1절에서 다룬 '고정 크기 벡터의 한계'")
print("→ Attention이 이 문제를 어떻게 푸는지는 다음 장에서 다룬다.")

---

## 8. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 13.2 | RNN 순전파 | PyTorch와 일치 ✓ |
| 13.3 | 그래디언트 감쇠 | 직접 측정 ✓ |
| **13.4** | **LSTM 게이트 4개 + c=0.924, h=0.462** | **일치** ✓ |
| 13.3 | 망각 게이트에 따른 기억 유지 | 표로 확인 ✓ |
| 13.6 | 순차 처리로 인한 속도 | 측정 ✓ |
| 13.6 | 정보 병목 | 확인 ✓ |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 입력 모양 | `(배치, 시점, 특성)` 3차원 |
| 출력 | `out`(모든 시점) / `h_n`(마지막) |
| LSTM 상태 | `h_n`과 `c_n` **두 개** |
| 파라미터 | RNN 1배 / GRU 3배 / LSTM 4배 |
| 시계열 분할 | **섞으면 안 됨** — 시간 순서 유지 |
| LSTM의 장점 | 감쇠를 없애는 것이 아니라 **조절 가능**하게 함 |

### 정직하게 짚어둘 것

3절 실험에서 RNN과 LSTM의 그래디언트 감쇠 차이가 크지 않게 나왔다.
**초기화 직후라 게이트가 아직 학습되지 않았기 때문**이다.

교과서적 설명은 "LSTM이 그래디언트 소실을 해결한다"이지만, 정확히는
**"해결할 수 있는 구조를 제공한다"**가 맞다. 실제로 그렇게 되려면 학습이 필요하고,
과제에 따라 효과가 다르다.

### 다음 장

**17. 강화학습 실습 — Q-Learning부터 DQN까지** — 이론편 18장. 정답이 없고 **보상만 주어지는** 상황을 다룬다.
이론편 14.3절에서 손으로 계산한 Q값 갱신(0 → 5.0 → 2.25)을 확인한다.